In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import requests
from pqdm.processes import pqdm

In [ ]:
df = pd.read_csv("/opt/gpudata/tcga/TCGA_Reports.csv")

In [13]:
case_ids = df["patient_filename"].str.split(".").str[0].to_list()

In [45]:
batch_size = 100

url = f"https://api.gdc.cancer.gov/cases"

params = {
    "filters": {
        "op":"in",
        "content": {
            "field": "cases.submitter_id",
            "value": ["replace_me!"]
        }
    },
    "fields": "submitter_id,project.project_id",
    "size": str(batch_size)
}

def get_project_id(case_id):
    if not isinstance(case_id, list):
        case_id = [case_id]
    params["filters"]["content"]["value"] = case_id
    response = requests.post(url, json=params)
    ret = dict()
    for hit in response.json()["data"]["hits"]:
        ret[hit["submitter_id"]] = hit["project"]["project_id"]
    return ret

rets = pqdm([case_ids[i:i+batch_size] for i in range(0, len(case_ids), batch_size)], get_project_id, n_jobs=2)
case_to_project = {k: v for batch in rets for k, v in batch.items()}
project_ids = [case_to_project[case_id] for case_id in case_ids]

QUEUEING TASKS | :   0%|          | 0/96 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/96 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/96 [00:00<?, ?it/s]

In [46]:
df["case_id"] = case_ids
df["project_id"] = project_ids

In [48]:
df.to_csv("TCGA_Reports.csv", index=False)